In [1]:
import os
from dotenv import load_dotenv

# This looks for the .env file and loads the variables into your environment
load_dotenv()

# Access the key
open_api_key = os.getenv("OPENAI_API_KEY")


## 1. Data Ingestion
Load KB articles and incidents from ServiceNow PDI export.

In [2]:
import pandas as pd
import os
df_kb = pd.read_csv('../data/raw/kb_knowledge.csv', encoding='latin-1')

df_incidents = pd.read_csv('../data/raw/incident.csv', encoding='latin-1')

print("KB columns:", df_kb.columns.tolist())
print("KB records:", len(df_kb))
print()
print("Incident columns:", df_incidents.columns.tolist())
print("Incident records:", len(df_incidents))

KB columns: ['number', 'short_description', 'description', 'text', 'kb_category', 'kb_knowledge_base']
KB records: 5076

Incident columns: ['number', 'short_description', 'description', 'close_notes', 'category', 'priority', 'assignment_group']
Incident records: 3698


## 2. Text Cleaning
Strip HTML tags and normalize text from KB article body fields.

In [3]:
from bs4 import BeautifulSoup
def clean_html(html_content: str) -> str:
    if pd.isna(html_content):
        return ""
    soup = BeautifulSoup(html_content, "html.parser")
    clean_text = soup.get_text().replace('*', '').replace('?', '').replace("'", "")
    clean_text = " ".join(clean_text.split())
    return clean_text
df_kb['clean_text'] = df_kb['text'].apply(clean_html)

In [4]:
def get_content(row):
    if len(row['clean_text']) > 0:
        return row['clean_text']
    elif not pd.isna(row['short_description']):
        return row['short_description'].strip()
    else:
        return ""

df_kb['content'] = df_kb.apply(get_content, axis=1)


## 3. Chunking
Split KB articles into paragraph-level chunks using RecursiveCharacterTextSplitter.

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def chunk_articles(text: str, article_id: str, chunk_size: int = 512, overlap: int = 50) -> list:
    if not text or len(text) == 0:
        return []
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len
    )
    
    chunks = splitter.split_text(text)
    
    return [
        {
            "text": chunk,
            "chunk_id": f"{article_id}_chunk_{i}",
            "article_id": article_id
        }
        for i, chunk in enumerate(chunks)
    ]

# Test on one article
sample_chunks = chunk_articles(df_kb['content'].iloc[0], df_kb['number'].iloc[0])
print(f"Chunks produced: {len(sample_chunks)}")
print(f"First chunk: {sample_chunks[0]['text'][:200]}")

/Users/gurukeerthana/applied-ai-prep/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Chunks produced: 11
First chunk: ServiceNow recognizes the importance of community goodwill and charitable contributions to nonprofit organizations. We are proud to support our employees personal passions by matching their contributi


In [6]:
all_kb_chunks = []

for row in df_kb.itertuples():
    
    chunks = chunk_articles(row.content, row.number) 
    
    for chunk in chunks:
        chunk['department_id'] = row.kb_category
        chunk['source_type'] = 'kb_article'
        chunk['knowledge_base'] = row.kb_knowledge_base
        
    all_kb_chunks.extend(chunks)
print(f"Total KB chunks: {len(all_kb_chunks)}")
print(f"Sample chunk metadata: {all_kb_chunks[0]}")

Total KB chunks: 24758
Sample chunk metadata: {'text': 'ServiceNow recognizes the importance of community goodwill and charitable contributions to nonprofit organizations. We are proud to support our employees personal passions by matching their contributions to qualified community or non-profit organizations. FAQs Refer to theServiceNows Giving at Now FAQs Company Match ServiceNow will provide a corporate match for employee contributions to qualified non-profit organizations at 100% of every dollar a regular employee donates up to $1,000 USD each calendar year.', 'chunk_id': 'KB0000560_chunk_0', 'article_id': 'KB0000560', 'department_id': 'Perks and Programs', 'source_type': 'kb_article', 'knowledge_base': 'Global People (HR) Knowledge'}


## 4. Incident Processing
Combine incident fields into single documents for indexing.

In [7]:
import re

def clean_incident_text(text: str) -> str:
    if not text or pd.isna(text):
        return ""
    
    # Remove common ServiceNow AI boilerplate phrases
    boilerplate_patterns = [
        r"Hello, I am ServiceNow AI.*?(?=Issue:|Steps:|Solution:|Best regards|$)",
        r"We are moving your Incident to Resolved.*?(?=Issue:|Steps:|Solution:|$)",
        r"If we have addressed your concerns.*?(?=Issue:|Steps:|Solution:|$)",
        r"you may accept the solution.*?(?=Issue:|Steps:|Solution:|$)",
        r"Best regards,\nServiceNow AI.*$",
        r"Best regards,\nServiceNow Support Engineer.*$",
    ]
    
    cleaned = text
    for pattern in boilerplate_patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.DOTALL | re.IGNORECASE)
    
    # Collapse extra whitespace
    cleaned = " ".join(cleaned.split())
    return cleaned

In [8]:
def safe_str(val):
    return str(val).strip() if pd.notna(val) else ""

def build_incident_doc(row) -> dict:
    # combine short_description + description + close_notes into one text
    # return dict with: text, incident_id, department_id, source_type, priority, category
   full_text = safe_str(row['short_description']) + " " + safe_str(row['description']) + " " + clean_incident_text(safe_str(row['close_notes']))
   full_text = " ".join(full_text.split())

   
   return {
        "text":  full_text,
        "incident_id": row.number,
        "department_id": str(row['assignment_group']).strip(),
        "source_type": "incident",
        "priority": row.priority,
        "category": row.category
    }



In [9]:
all_incident_docs = []

for _, row in df_incidents.iterrows():
    doc = build_incident_doc(row)
    all_incident_docs.append(doc)

print(f"Total incident docs: {len(all_incident_docs)}")
print(f"Sample metadata: department={all_incident_docs[0]['department_id']}, priority={all_incident_docs[0]['priority']}")

Total incident docs: 3698
Sample metadata: department=DT-GPS, priority=4 - Low


## 5. Vector Store Indexing
Embed and index all documents into ChromaDB with multi-tenant metadata.

In [258]:
import chromadb
from openai import OpenAI

openai_client = OpenAI(api_key=open_api_key)
chroma_client = chromadb.PersistentClient(path="../data/processed/chromadb")

kb_collection = chroma_client.get_or_create_collection(name="kb_articles")
incident_collection = chroma_client.get_or_create_collection(name="incidents")

print("ChromaDB collections created")
print("KB collection:", kb_collection.name)
print("Incident collection:", incident_collection.name)

ChromaDB collections created
KB collection: kb_articles
Incident collection: incidents


In [261]:
import time

def embed_and_index(chunks: list, collection, batch_size: int = 100):
    total = len(chunks)
    
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        
        texts = [c['text'] for c in batch]
        ids = [c['chunk_id'] if 'chunk_id' in c else c['incident_id'] for c in batch]
        metadatas = [{k: v for k, v in c.items() if k != 'text'} for c in batch]
        
        while True:
            try:
                response = openai_client.embeddings.create(
                    input=texts,
                    model="text-embedding-3-small"
                )
                break
            except Exception as e:
                print(f"Rate limit hit, waiting 10 seconds... {e}")
                time.sleep(10)
        
        embeddings = [r.embedding for r in response.data]
        
        collection.upsert(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        print(f"Indexed {min(i + batch_size, total)}/{total}")

In [175]:
seen_ids = set()
unique_kb_chunks = []

for chunk in all_kb_chunks:
    if chunk['chunk_id'] not in seen_ids:
        seen_ids.add(chunk['chunk_id'])
        unique_kb_chunks.append(chunk)

print(f"After dedup: {len(unique_kb_chunks)} chunks")

After dedup: 24747 chunks


In [249]:
kb_collection = chroma_client.get_or_create_collection(name="kb_articles")
incident_collection = chroma_client.get_or_create_collection(name="incidents")


In [253]:
embed_and_index(unique_kb_chunks, kb_collection, batch_size=500)

Indexed 500/24747
Indexed 1000/24747
Indexed 1500/24747
Indexed 2000/24747
Indexed 2500/24747
Indexed 3000/24747
Indexed 3500/24747
Indexed 4000/24747
Indexed 4500/24747
Indexed 5000/24747
Indexed 5500/24747
Indexed 6000/24747
Indexed 6500/24747
Indexed 7000/24747
Indexed 7500/24747
Indexed 8000/24747
Indexed 8500/24747
Indexed 9000/24747
Indexed 9500/24747
Indexed 10000/24747
Indexed 10500/24747
Indexed 11000/24747
Indexed 11500/24747
Indexed 12000/24747
Indexed 12500/24747
Indexed 13000/24747
Indexed 13500/24747
Indexed 14000/24747
Indexed 14500/24747
Indexed 15000/24747
Indexed 15500/24747
Indexed 16000/24747
Indexed 16500/24747
Indexed 17000/24747
Indexed 17500/24747
Indexed 18000/24747
Indexed 18500/24747
Indexed 19000/24747
Indexed 19500/24747
Indexed 20000/24747
Indexed 20500/24747
Indexed 21000/24747
Indexed 21500/24747
Indexed 22000/24747
Indexed 22500/24747
Indexed 23000/24747
Indexed 23500/24747
Indexed 24000/24747
Indexed 24500/24747
Indexed 24747/24747


In [259]:
embed_and_index(all_incident_docs, incident_collection, batch_size=100)

Indexed 100/3698
Indexed 200/3698
Indexed 300/3698
Indexed 400/3698
Indexed 500/3698
Indexed 600/3698
Indexed 700/3698
Indexed 800/3698
Indexed 900/3698
Indexed 1000/3698
Indexed 1100/3698
Indexed 1200/3698
Indexed 1300/3698
Indexed 1400/3698
Indexed 1500/3698
Indexed 1600/3698
Indexed 1700/3698
Indexed 1800/3698
Indexed 1900/3698
Indexed 2000/3698
Indexed 2100/3698
Indexed 2200/3698
Indexed 2300/3698
Indexed 2400/3698
Indexed 2500/3698
Indexed 2600/3698
Indexed 2700/3698
Indexed 2800/3698
Indexed 2900/3698
Indexed 3000/3698
Indexed 3100/3698
Indexed 3200/3698
Indexed 3300/3698
Indexed 3400/3698
Indexed 3500/3698
Indexed 3600/3698
Indexed 3698/3698


In [262]:
print(f"KB collection count: {kb_collection.count()}")
print(f"Incident collection count: {incident_collection.count()}")

KB collection count: 24747
Incident collection count: 3698


## 6. Retrieval
Query ChromaDB using OpenAI embeddings with department-scoped filtering.

In [180]:
def query_collection(collection, query: str, n_results: int = 3, where: dict = None):
    response = openai_client.embeddings.create(
        input=[query],
        model="text-embedding-3-small"
    )
    query_embedding = response.data[0].embedding
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where
    )
    return results

# Test
results = query_collection(
    kb_collection,
    query="how to reset password",
    n_results=3,
    where={"department_id": "Software"}
)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['department_id'], "—", doc[:100])
    print()

Software — installed using admin credentials. If you only need the SaaS component, you can tell Level 1 Support

Software — or mobile client (make sure youre signed in). For audio specifically, join through your computer or 

Software — Software Specific Usage Instructions Workshop is a SaaS application available here or through the Ok



In [181]:
results = query_collection(
    incident_collection,
    query="Okta SSO access denied",
    n_results=3,
    where={"department_id": "DT-GPS"}
)
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['department_id'], "—", doc[:150])
    print()

DT-GPS — Okta | Unable to login error | ZV - POD Validated Hello Team, I am unable to login ServiceNow Okta getting error. Please find the screen short for you

DT-GPS — Issue with: Okta (Internal) Unable to access ServiceNow due to okta failure Hello, I am ServiceNow AI and I am assisting with your incident. We are mo

DT-GPS — Issue with: Okta (Internal) ServiceNow SSO login issues Hello, I am ServiceNow AI and I am helping on your incident. We are moving your Incident to Re



In [ ]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')



config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## 7. Multi-Agent Graph
LangGraph graph with Router, KB Worker, Incident Worker, Synthesizer, and Security nodes.

In [226]:
from typing import Literal
from pydantic import BaseModel, Field

# 1. Exact keys from your registry
DepartmentLiteral = Literal[
    "IT Software", 
    "DT-GPS", 
    "Global People Live Chat Agents", 
    "WPS - Badging",
    "Other" # Add an 'Other' as a fallback instead of 'None'
]

class RouteAction(BaseModel):
    # 2. Strict source types - removed 'general'
    source_type: Literal["kb_article", "incident", "out_of_scope"] = Field(
        ..., 
        description="Must be 'kb_article' for how-to, 'incident' for problems, or 'out_of_scope' for non-work queries."
    )
    
    # 3. REQUIRED field. Removed Optional and None default.
    department_id: DepartmentLiteral = Field(
        ..., 
        description="You MUST pick the best department. For Okta/VPN/Login, you MUST pick 'DT-GPS'."
    )
    
    refined_query: str = Field(..., description="The search query.")

In [183]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
from typing import Literal, Optional
from pydantic import BaseModel, Field


class AgentState(TypedDict):
    messages: Annotated[list, add_messages] 
    route: Optional[RouteAction]
    kb_results: Annotated[list[str], operator.add]
    incident_results: Annotated[list[str], operator.add]
    final_answer: str
    verified_tenant_id:str
    security_violation: bool

In [ ]:
from langchain_openai import ChatOpenAI
# Set this BEFORE initializing the model
os.environ["OPENAI_API_KEY"] = open_api_key
router_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)      # cheap, fast
synthesizer_llm = ChatOpenAI(model="gpt-4o", temperature=0)        # powerful, expensive

In [185]:
DEPARTMENT_REGISTRY = {
    "IT Software": {
        "description": "Issues with installed applications, OS updates, and developer tools.",
        "keywords": ["Git", "IntelliJ", "VS Code", "Python", "Java", "Slack", "Office 365"]
    },
    "DT-GPS": {
        "description": "Global Provisioning Services. Primary silo for access and identity.",
        "keywords": ["Okta", "VPN", "SSO", "Zscaler", "Login", "Password Reset", "Access Denied"]
    },
    "Global People Live Chat Agents": {
        "description": "Human Resources and employee perks.",
        "keywords": ["Matching Gifts", "Benefits", "Gym Reimbursement", "Payroll", "HR"]
    },
    "WPS - Badging": {
        "description": "Workplace Services and physical office facilities.",
        "keywords": ["Badge access", "Office temp", "Desk booking", "Physical Maintenance"]
    }
}

In [227]:
def router_node(state: AgentState):
    # This creates a simple list: "IT Software, DT-GPS, ..."
    allowed_depts = ", ".join(DEPARTMENT_REGISTRY.keys())

    # This builds a structured block of text for the LLM to study
    registry_parts = []
    for dept, info in DEPARTMENT_REGISTRY.items():
        part = f"DEPT: {dept}\nDESCRIPTION: {info['description']}\nKEYWORDS: {', '.join(info['keywords'])}"
        registry_parts.append(part)
    
    registry_text = "\n\n".join(registry_parts)
    
    system_prompt = f"""
You are a ServiceNow Router. Your task is to map queries to the correct department.

1. ALLOWED DEPARTMENTS (You MUST pick one of these keys):
{allowed_depts}

2. MAPPING TAXONOMY (Use keywords to find the match):
{registry_text}

3. CLASSIFICATION STEPS:
   - Identify the tool in the query (e.g., 'Okta').
   - Find which DEPT has that tool in its keywords.
   - Return the EXACT key from the ALLOWED DEPARTMENTS list.
   - If 'Okta' or 'VPN' is found, you are FORBIDDEN from returning None; you MUST return 'DT-GPS'.
"""
    
    structured_router = router_llm.with_structured_output(RouteAction, method="function_calling")
    
    user_message = state["messages"][-1].content
    decision = structured_router.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ])
    
    # Debug print to verify the fix
    print(f"--- ROUTING DECISION: {decision.source_type} | {decision.department_id} ---")
    
    return {"route": decision}

In [215]:
def kb_worker_node(state: AgentState):
    """
    Retrieves KB articles with a strict Identity Cross-Check.
    """
    route = state.get("route")
    if not route:
        return {"kb_results": ["Error: No routing instructions found."]}
        
    query = route.refined_query
    
    # 1. What the AI thinks the user wants (Intent)
    requested_dept = route.department_id 
    
    # 2. What the System knows the user is allowed to see (ACL)
    authorized_dept = state.get("verified_tenant_id", "UNKNOWN_TENANT")
    
    # 3. THE CROSS-CHECK (Your Architectural Move)
    if requested_dept and requested_dept != authorized_dept:
        print(f"--- SECURITY BLOCK: User authorized for {authorized_dept} attempted to query {requested_dept} ---")
        
        # We short-circuit the database call entirely to save compute and enforce security
        security_msg = f"Security Violation: You are authorized for '{authorized_dept}' but your query requires access to '{requested_dept}'. Access Denied."
        return {
            "security_violation": True, 
            "kb_results": [f"Violation: {requested_dept} vs {authorized_dept}"]
        }
    
    print(f"--- KB WORKER: Authorized search in '{authorized_dept}' for '{query}' ---")
    
    # 4. If they match, execute the safe search
    try:
        search_results = query_collection(
            collection=kb_collection,
            query=query,
            n_results=3,
            where={"department_id": authorized_dept} 
        )
        candidates = search_results.get("documents", [[]])[0]

        if not candidates:
            return {"kb_results": [], "security_violation": False}

        # STAGE 2: RERANKING
        # Use the cross-encoder to find the best 3 matches for the specific query
        pairs = [[query, doc] for doc in candidates]
        scores = reranker_model.predict(pairs)
        
        # Sort and prune to the top 3 for the synthesizer
        scored_docs = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        top_3_docs = [doc for score, doc in scored_docs[:3]]
        
        print(f"--- RERANKER: Evaluated {len(candidates)} incidents, selected top 3 ---")
        
    except Exception as e:
        top_3_docs = [f"Error retrieving database records: {str(e)}"]
    
    return {"security_violation": False, "kb_results": top_3_docs}

In [216]:
def incident_worker_node(state: AgentState):
    """
    Retrieves Incidents using Two-Stage Retrieval:
    1. Broad Recall: Vector search + metadata filter (n=25)
    2. High-Precision: Cross-Encoder reranking
    """
    route = state.get("route")
    if not route:
        return {"incident_results": ["Error: No routing instructions found."]}
        
    query = route.refined_query
    authorized_dept = state.get("verified_tenant_id", "UNKNOWN_TENANT")
    requested_dept = route.department_id 
    
    # 1. Security Cross-Check
    if requested_dept and requested_dept != authorized_dept:
        print(f"--- SECURITY BLOCK (INCIDENT): Access Denied for {requested_dept} ---")
        return {
            "security_violation": True, 
            "incident_results": [f"Violation: {requested_dept} vs {authorized_dept}"]
        }
    
    print(f"--- INCIDENT WORKER: Multi-Stage Search in '{authorized_dept}' ---")
    
    try:
        # STAGE 1: BROAD RECALL
        # Pulling 25 candidates to bypass the 'Top-K' metadata filter bottleneck
        search_results = query_collection(
            collection=incident_collection,
            query=query,
            n_results=25,
            where={"department_id": authorized_dept} 
        )
        candidates = search_results.get("documents", [[]])[0]
        
        if not candidates:
            return {"incident_results": [], "security_violation": False}

        # STAGE 2: RERANKING
        # Use the cross-encoder to find the best 3 matches for the specific query
        pairs = [[query, doc] for doc in candidates]
        scores = reranker_model.predict(pairs)
        
        # Sort and prune to the top 3 for the synthesizer
        scored_docs = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        top_3_docs = [doc for score, doc in scored_docs[:3]]
        
        print(f"--- RERANKER: Evaluated {len(candidates)} incidents, selected top 3 ---")
        
    except Exception as e:
        top_3_docs = [f"Error in incident retrieval: {str(e)}"]
    
    return {"security_violation": False, "incident_results": top_3_docs}

In [217]:
def refusal_node(state: AgentState):
    # refusal message that doesn't leak system prompts
    explanation = state["route"].refined_query if state["route"] else "I cannot assist with this."
    
    msg = f"I'm sorry, I'm authorized only for ServiceNow ITSM and HR tasks. {explanation}"
    
    return {"final_answer": msg}

In [231]:
def synthesizer_node(state: AgentState):
    # Use a set of keys to be more robust
    kb_data = state.get("kb_results", [])
    inc_data = state.get("incident_results", [])
    
    # Pre-process: Label the data so the LLM understands the 'Signal'
    context_list = []
    for doc in kb_data:
        context_list.append(f"[SOURCE: Knowledge Base]\n{doc}")
    for doc in inc_data:
        context_list.append(f"[SOURCE: Past Incident Resolution]\n{doc}")
    
    all_context = "\n\n---\n\n".join(context_list)
    
    # Direct and Critical Prompting
    prompt = f"""
    You are a ServiceNow Support Expert. Your goal is to provide a technical solution to the user's query using the provided context.
    
    CONTEXT:
    {all_context if all_context.strip() else "NO DATA FOUND"}
    
   USER QUERY: {state['messages'][-1].content}
    
    INSTRUCTIONS:
    1. If the CONTEXT contains steps to resolve the issue (even if they are from past Incidents), present them as the SOLUTION.
    2. DO NOT say "I couldn't find a solution" if there are troubleshooting steps in the context.
    3. Use an authoritative, helpful tone. 
    4. If the context is 'NO DATA FOUND', only then state that you couldn't find a record.
    """
    
    response = synthesizer_llm.invoke(prompt)
    return {"final_answer": response.content}

In [219]:
def security_alert_node(state: AgentState):
    """
    Uses a cheap, low-latency model to format security rejections.
    """
    # Grab the violation details from the worker
    violation_details = state.get("kb_results", [""])[0] 
    
    prompt = f"""
    You are a strict Enterprise Security Assistant. 
    The user attempted to access restricted data. 
    Write a brief, polite, but firm 'Access Denied' message.
    Details: {violation_details}
    """
    
    # Use the cheap LLM!
    response = router_llm.invoke(prompt) 
    
    return {"final_answer": response.content}

In [192]:
def check_worker_security(state: AgentState):
    """
    Checks if the worker tripped the security alarm.
    """
    if state.get("security_violation"):
        return "security_alert"
    return "synthesizer"

In [193]:
def should_continue(state: AgentState):
    routing_result = state.get("route")
    if not routing_result or routing_result.source_type == "out_of_scope":
     return "refusal_node"
    # Parallel Branching (Fan-out): Trigger specific workers.
    if routing_result.source_type == "kb_article":
        return "kb_worker"
    
    if routing_result.source_type == "incident":
        return "incident_worker"
    
    # If it's a general greeting or basic query, go straight to synthesis.
    return "synthesizer"

In [232]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

# 1. ADD NODES (Strict Naming: String ID exactly matches Function Name)
workflow.add_node("router_node", router_node)
workflow.add_node("kb_worker_node", kb_worker_node)
workflow.add_node("incident_worker_node", incident_worker_node)
workflow.add_node("refusal_node", refusal_node)
workflow.add_node("synthesizer_node", synthesizer_node)
workflow.add_node("security_alert_node", security_alert_node)

# 2. SET ENTRY POINT (Standardized to 'router_node')
workflow.add_edge(START, "router_node")

# 3. ROUTER CONDITIONAL EDGES
workflow.add_conditional_edges(
    "router_node",
    should_continue,
    {
        # The key is what `should_continue` returns.
        # The value is our strictly named target node.
        "kb_worker": "kb_worker_node",
        "incident_worker": "incident_worker_node",
        "refusal_node": "refusal_node",
        "synthesizer": "synthesizer_node"
    }
)

# 4. KB WORKER CONDITIONAL EDGES
workflow.add_conditional_edges(
    "kb_worker_node", # The source MUST match the node name defined in step 1
    check_worker_security,
    {
        "security_alert": "security_alert_node", 
        "synthesizer": "synthesizer_node"        
    }
)

# 5. INCIDENT WORKER CONDITIONAL EDGES
workflow.add_conditional_edges(
    "incident_worker_node", # The source MUST match the node name defined in step 1
    check_worker_security,
    {
        "security_alert": "security_alert_node", 
        "synthesizer": "synthesizer_node"        
    }
)

# 6. EXIT POINTS (Standardized names)
workflow.add_edge("synthesizer_node", END)       
workflow.add_edge("refusal_node", END)
workflow.add_edge("security_alert_node", END)    

# 7. COMPILE
app = workflow.compile()
print("Graph compiled successfully!")

Graph compiled successfully!


## 8. Demo
End-to-end agent query demonstrating the full pipeline.

In [233]:
inputs = {
    "messages": [("user", "I can't access Okta, how do I fix it?")],
    "verified_tenant_id": "DT-GPS", # Must match the 'Actual Departments' list exactly
    "kb_results": [],
    "incident_results": [],
    "security_violation": False
}

for output in app.stream(inputs):
    print(output)

--- ROUTING DECISION: incident | DT-GPS ---
{'router_node': {'route': RouteAction(source_type='incident', department_id='DT-GPS', refined_query="I can't access Okta, how do I fix it?")}}
--- INCIDENT WORKER: Multi-Stage Search in 'DT-GPS' ---
--- RERANKER: Evaluated 25 incidents, selected top 3 ---
{'incident_worker_node': {'security_violation': False, 'incident_results': ['Okta | cannot access docs.servicenow.com cannot access docs.servicenow.com Hello, I am ServiceNow AI and I am helping on your incident. We are moving your Incident to Resolved as we believe the information provided below will resolve your issue. If we have addressed your concerns, you may accept the solution to close this incident or reject the solution if it does not answer the original question raised with this ticket. At any time in the Resolved state, you may add additional questions or updates. Issue: Unable to access docs.servicenow.com via Okta (Internal). Unexpected/Actual Behavior: Access to docs.servicenow

In [210]:
# Run this in a new cell
results = incident_collection.get(
    where={"department_id": "DT-GPS"},
    include=["metadatas"]
)
print(f"Total documents found for 'DT-GPS': {len(results['ids'])}")
if len(results['ids']) > 0:
    print(f"Sample metadata: {results['metadatas'][0]}")

Total documents found for 'DT-GPS': 1689
Sample metadata: {'source_type': 'incident', 'department_id': 'DT-GPS', 'incident_id': 'INC1637267', 'priority': '4 - Low', 'category': 'Enterprise IAM'}
